# Deep learning recommender

Author: Masha Khanina

Reviewer: 

Description: making movie recommendations system by using two-tower model for retrieval + mlp with early crossing for ranking recommendations.

Infinity Labs R&D AI #5

In [1]:
import copy
from collections import defaultdict

import duckdb
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import ndcg_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [2]:
with duckdb.connect("/home/maria-khanina/mkgit/movies_project/data/database.duckdb", read_only=True) as db_conn:
    ratings_df = db_conn.execute("""
        SELECT *
        FROM ratings
            INNER JOIN movies USING (movie_id)
            INNER JOIN users USING (user_id)
    """).df()

In [3]:
ratings_df.head()

,user_id,movie_id,rating,timestamp,title,genres,gender,age,occupation,zip_code
0,793,2407,2,975421808,Cocoon (1985),Comedy|Sci-Fi,F,35,14,37043
1,793,2268,4,975422633,"Few Good Men, A (1992)",Crime|Drama,F,35,14,37043
2,793,933,3,975423990,To Catch a Thief (1955),Comedy|Romance|Thriller,F,35,14,37043
3,793,3217,2,975423165,"Star Is Born, A (1937)",Drama,F,35,14,37043
4,793,1617,3,975420951,L.A. Confidential (1997),Crime|Film-Noir|Mystery|Thriller,F,35,14,37043


In [4]:
ratings_df["age"].value_counts()

age
25    395556
35    199003
18    183536
45     83633
50     72490
56     38780
1      27211
Name: count, dtype: int64

In [5]:
genres_encoded = ratings_df["genres"].str.get_dummies(sep="|")
genres_cols = genres_encoded.columns.to_list()
genres_num = len(genres_cols)
ratings_df = pd.concat([ratings_df, genres_encoded], axis=1)

In [6]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

cat_cols = ["gender"]

ratings_df[cat_cols] = encoder.fit_transform(ratings_df[cat_cols]) + 1
ratings_df["occupation"] = ratings_df["occupation"] + 1

In [7]:
max_user = int(ratings_df["user_id"].max())
max_movie = int(ratings_df["movie_id"].max())
max_age = int(ratings_df["age"].max())
max_gender = int(ratings_df["gender"].max())
max_job = int(ratings_df["occupation"].max())

In [8]:
class Ratings(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        genres_cols: list[str],
        is_training: bool = True,
    ) -> None:
        self.user_ids = torch.tensor(df["user_id"].values, dtype=torch.long)
        self.movie_ids = torch.tensor(df["movie_id"].values, dtype=torch.long)
        self.ages = torch.tensor(df["age"].values, dtype=torch.long)
        self.genders = torch.tensor(df["gender"].values, dtype=torch.long)
        self.jobs = torch.tensor(df["occupation"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["normalized_rating"].values, dtype=torch.float32)
        self.genres = torch.tensor(df[genres_cols].values, dtype=torch.float32)
        self.is_training = is_training

    def __len__(self) -> int:
        return len(self.user_ids)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        
        # user_id = int(row["user_id"])
        # if self.is_training and torch.rand(1).item() < 0.05:
            # user_id = 0
            
        return {
            "user_id": self.user_ids[idx],
            "movie_id": self.movie_ids[idx],
            "age": self.ages[idx],
            "gender": self.genders[idx],
            "job": self.jobs[idx],
            "genres": self.genres[idx], 
            "rating": self.ratings[idx],
        }

In [9]:
class BaseRanker(nn.Module):
    def __init__(
        self,
        users_num: int,
        movies_num: int,
        add_interactions: bool = False,
        lin_layers_dims: list[int] = [128, 64, 32],
        dropout_ratio=0.2,
        user_dim=16,
        movie_dim=16,
        extra_input_dim=0,
    ) -> None:
        super().__init__()
        
        self.add_interactions = add_interactions
        
        self.user_embedding = nn.Embedding(
            num_embeddings=users_num + 1,
            embedding_dim=user_dim,
        )
        self.movie_embedding = nn.Embedding(
            num_embeddings=movies_num + 1,
            embedding_dim=movie_dim,
        )

        input_dim = user_dim + movie_dim + extra_input_dim
        
        if self.add_interactions:
            assert user_dim == movie_dim
            input_dim += user_dim

        self.mlp = self._build_mlp(input_dim, lin_layers_dims, dropout_ratio)

    def _build_mlp(
        self,
        input_dim: int, 
        lin_layers_dims: list[int],
        dropout_ratio: float,
    ) -> nn.Sequential:
        layers = []
        curr_dim = input_dim
        
        for ind, dim in enumerate(lin_layers_dims):
            layers.append(nn.Linear(curr_dim, dim))
            layers.append(nn.SiLU())
            layers.append(nn.BatchNorm1d(dim))
            
            if ind == 0:
                layers.append(nn.Dropout(p=dropout_ratio))
                
            curr_dim = dim
            
        layers.append(nn.Linear(curr_dim, 1))
        return nn.Sequential(*layers)

    def _get_base_features(
        self,
        user_ids: torch.Tensor,
        movie_ids: torch.Tensor,
    ) -> list[torch.Tensor]:
        user_vector = self.user_embedding(user_ids)
        movie_vector = self.movie_embedding(movie_ids)
        
        features = [user_vector, movie_vector]
        
        if self.add_interactions:
            features.append(user_vector * movie_vector)
            
        return features

    def forward(
        self,
        user_ids: torch.Tensor,
        movie_ids: torch.Tensor,
    ) -> torch.Tensor:
        
        features = self._get_base_features(user_ids, movie_ids)
        cat_features = torch.cat(features, dim=1)
        
        return self.mlp(cat_features).squeeze()


class RankerWithFeatures(BaseRanker):
    def __init__(
        self,
        users_num: int,
        movies_num: int,
        ages_num: int,
        genders_num: int,
        jobs_num: int,
        genres_num: int,
        add_interactions: bool = False,
        lin_layers_dims: list[int] = [128, 64, 32],
        dropout_ratio=0.2,
        user_dim=16,
        movie_dim=16,
        age_dim=3,
        gender_dim=2,
        job_dim=4,
        genre_dim=4,
    ) -> None:
        
        extra_dim = age_dim + gender_dim + job_dim + genre_dim

        super().__init__(
            users_num=users_num,
            movies_num=movies_num,
            add_interactions=add_interactions,
            lin_layers_dims=lin_layers_dims,
            dropout_ratio=dropout_ratio,
            user_dim=user_dim,
            movie_dim=movie_dim,
            extra_input_dim=extra_dim,
        )

        self.job_embedding = nn.Embedding(
            num_embeddings=jobs_num + 1,
            embedding_dim=job_dim,
        )
        self.age_embedding = nn.Embedding(
            num_embeddings=ages_num + 1,
            embedding_dim=age_dim,
        )
        self.gender_embedding = nn.Embedding(
            num_embeddings=genders_num + 1,
            embedding_dim=gender_dim,
        )
        self.genre_projector = nn.Linear(genres_num, genre_dim, bias=False)

    def forward(
        self,
        user_ids: torch.Tensor,
        movie_ids: torch.Tensor,
        ages: torch.Tensor,
        genders: torch.Tensor,
        jobs: torch.Tensor,
        genres: torch.Tensor,
    ) -> torch.Tensor:
        
        features = self._get_base_features(user_ids, movie_ids)
        
        features.extend([
            self.age_embedding(ages),
            self.gender_embedding(genders),
            self.job_embedding(jobs),
            self.genre_projector(genres),
        ])
        
        cat_features = torch.cat(features, dim=1)
        
        return self.mlp(cat_features).squeeze()

In [10]:
def move_tensors(
        var_key_map: dict[str, str],
        batch: dict[str, torch.Tensor],
        device: torch.device,
) -> dict[str, torch.Tensor]:
    feature_tensors: dict[str, torch.Tensor] = {}
    for var_name, col_name in var_key_map.items():
        feature_tensors[var_name] = batch[col_name].to(device)
    return feature_tensors

def _run_epoch(
    model: nn.Module,
    dataloader: DataLoader[dict[str, torch.Tensor]],
    var_key_map: dict[str, str],
    criterion: nn.Module,
    device: torch.device,
    is_train: bool,
    optimizer: optim.Optimizer | None = None,
) -> float:
    if is_train:
        model.train()
    else:
        model.eval()
    
    epoch_loss = 0.0
    dataset_len = 0
    
    with torch.set_grad_enabled(is_train):
        for batch in dataloader:
            feature_tensors = move_tensors(var_key_map, batch, device)

            y_true = batch["rating"].to(device).float()
            y_pred = model(**feature_tensors)
            
            loss = criterion(y_pred, y_true)
            
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
            epoch_loss += loss.item() * feature_tensors["user_ids"].size(0)
            dataset_len += feature_tensors["user_ids"].size(0)
            
    return epoch_loss / dataset_len


def train_ranker(
    model: nn.Module,
    train_loader: DataLoader[dict[str, torch.Tensor]],
    val_loader: DataLoader[dict[str, torch.Tensor]],
    var_key_map: dict[str, str],
    epochs=10_000,
    learning_rate=0.001,
    patience=10,
    tol=1e-4,
    weight_decay=1e-5,
) -> None:
    """if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")"""
    device = torch.device("cpu")
    model = model.to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    
    best_val_loss = np.inf
    to_beat_loss = np.inf
    no_improve_count = 0
    best_weights = None
    
    for epoch in range(epochs):
        avg_train_loss = _run_epoch(
            model=model,
            dataloader=train_loader,
            var_key_map=var_key_map,
            criterion=criterion,
            device=device,
            is_train=True,
            optimizer=optimizer,
        )
        
        avg_val_loss = _run_epoch(
            model=model,
            dataloader=val_loader,
            var_key_map=var_key_map,
            criterion=criterion,
            device=device,
            is_train=False,
        )
        
        print(
            f"Epoch {epoch+1}/{epochs}, "
            f"Train RMSE: {np.sqrt(avg_train_loss):.4f}, "
            f"Val RMSE: {np.sqrt(avg_val_loss):.4f}."
        )
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_weights = copy.deepcopy(model.state_dict())

        if avg_val_loss < (to_beat_loss - tol):
            to_beat_loss = avg_val_loss
            no_improve_count = 0
        else:
            no_improve_count += 1
            
        if no_improve_count >= patience:
            break
    
    if best_weights is not None:
        model.load_state_dict(best_weights)
        print(f"\nBest weights restored. Val RMSE: {np.sqrt(best_val_loss)}")

In [11]:
def per_var_split_mask(
        df_sorted: pd.DataFrame,
        var: str,
        max_test_ratio=0.1,
        test_max_per_var=8,
        test_min_per_var=3,
):
    """
        Prepare splitting for given dataframe such as each unique value of var 
        has max_test_ratio ratio of records in test set capped at test_max_per_var.
    """
    per_var_count = df_sorted[var].map(df_sorted[var].value_counts())
    per_var_ranks = df_sorted.groupby(var).cumcount(ascending=False)
    per_var_test_size = np.minimum(test_max_per_var, max_test_ratio * per_var_count)
    per_var_test_size = np.maximum(per_var_test_size, test_min_per_var)
    test_mask = per_var_ranks < per_var_test_size

    return test_mask

In [12]:
def per_var_temporal_split(
        X: pd.DataFrame,
        var="user_id",
        time_col="timestamp",
        max_test_ratio=0.1,
        test_max_per_var=8,
        test_min_per_var=3,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
        Split X and y to train and test such as for each value of var
        all test records have greater time_col values than train records,
        and each value of var receives max_test_ratio of records in test,
        capped at test_max_per_var.
    """
    sort_indices = X.sort_values(by=[var, time_col]).index
    X_sorted = X.loc[sort_indices]
    test_mask = per_var_split_mask(
        X_sorted, 
        var, 
        max_test_ratio, 
        test_max_per_var,
        test_min_per_var,
    )
    return X_sorted[~test_mask], X_sorted[test_mask]

In [13]:
train_all_df, test_df = per_var_temporal_split(
    X=ratings_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

In [14]:
train_big_df, outer_val_df = per_var_temporal_split(
    train_all_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

In [15]:
train_df, val_df = per_var_temporal_split(
    train_big_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

In [16]:
rating_mean = train_df["rating"].mean()
users_rating_means = train_df.groupby("user_id")["rating"].mean()

for df in [train_df, val_df, outer_val_df, test_df]:
    df["user_mean"] = df["user_id"].map(users_rating_means).fillna(rating_mean)
    df["normalized_rating"] = df["rating"] - df["user_mean"]

In [28]:
var_key_map = {
    "user_ids": "user_id",
    "movie_ids": "movie_id",
    "ages": "age",
    "genders": "gender",
    "jobs": "job",
    "genres": "genres",
}

In [306]:
train_dataset = Ratings(train_df, genres_cols, is_training=True)
val_dataset = Ratings(val_df, genres_cols, is_training=False)

train_loader = DataLoader(
    train_dataset, 
    batch_size=1024, 
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=1024, 
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

ranker_with_interactions = RankerWithFeatures(
    users_num=max_user,
    movies_num=max_movie,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
    genres_num=genres_num,
    add_interactions=True,
)

train_ranker(
    model=ranker_with_interactions,
    train_loader=train_loader,
    val_loader=val_loader,
    var_key_map=var_key_map,
    epochs=10_000,
    learning_rate=0.002,
)

Epoch 1/10000, Train RMSE: 0.9723, Val RMSE: 0.9673.
Epoch 2/10000, Train RMSE: 0.9254, Val RMSE: 0.9599.
Epoch 3/10000, Train RMSE: 0.9133, Val RMSE: 0.9502.
Epoch 4/10000, Train RMSE: 0.8999, Val RMSE: 0.9386.
Epoch 5/10000, Train RMSE: 0.8906, Val RMSE: 0.9338.
Epoch 6/10000, Train RMSE: 0.8840, Val RMSE: 0.9280.
Epoch 7/10000, Train RMSE: 0.8791, Val RMSE: 0.9239.
Epoch 8/10000, Train RMSE: 0.8750, Val RMSE: 0.9221.
Epoch 9/10000, Train RMSE: 0.8714, Val RMSE: 0.9218.
Epoch 10/10000, Train RMSE: 0.8680, Val RMSE: 0.9196.
Epoch 11/10000, Train RMSE: 0.8650, Val RMSE: 0.9178.
Epoch 12/10000, Train RMSE: 0.8619, Val RMSE: 0.9154.
Epoch 13/10000, Train RMSE: 0.8589, Val RMSE: 0.9151.
Epoch 14/10000, Train RMSE: 0.8563, Val RMSE: 0.9152.
Epoch 15/10000, Train RMSE: 0.8536, Val RMSE: 0.9156.
Epoch 16/10000, Train RMSE: 0.8512, Val RMSE: 0.9149.
Epoch 17/10000, Train RMSE: 0.8483, Val RMSE: 0.9149.
Epoch 18/10000, Train RMSE: 0.8458, Val RMSE: 0.9143.
Epoch 19/10000, Train RMSE: 0.8433, V

In [ ]:
big_ranker_no_interactions = RankerWithFeatures(
    users_num=max_user,
    movies_num=max_movie,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
    genres_num=genres_num,
    add_interactions=False,
    dropout_ratio=0,
    user_dim=32,
    movie_dim=32,
    age_dim=4,
    gender_dim=1,
    job_dim=8,
    genre_dim=8,
)


train_ranker(
    model=big_ranker_no_interactions,
    train_loader=train_loader,
    val_loader=val_loader,
    var_key_map=var_key_map,
    epochs=10_000,
    learning_rate=0.002,
)

Epoch 1/10000, Train RMSE: 0.9509, Val RMSE: 0.9624.
Epoch 2/10000, Train RMSE: 0.9147, Val RMSE: 0.9500.
Epoch 3/10000, Train RMSE: 0.8955, Val RMSE: 0.9375.
Epoch 4/10000, Train RMSE: 0.8821, Val RMSE: 0.9293.
Epoch 5/10000, Train RMSE: 0.8725, Val RMSE: 0.9266.
Epoch 6/10000, Train RMSE: 0.8645, Val RMSE: 0.9234.
Epoch 7/10000, Train RMSE: 0.8571, Val RMSE: 0.9218.
Epoch 8/10000, Train RMSE: 0.8498, Val RMSE: 0.9220.
Epoch 9/10000, Train RMSE: 0.8425, Val RMSE: 0.9203.
Epoch 10/10000, Train RMSE: 0.8355, Val RMSE: 0.9214.
Epoch 11/10000, Train RMSE: 0.8286, Val RMSE: 0.9238.
Epoch 12/10000, Train RMSE: 0.8219, Val RMSE: 0.9261.
Epoch 13/10000, Train RMSE: 0.8153, Val RMSE: 0.9265.
Epoch 14/10000, Train RMSE: 0.8087, Val RMSE: 0.9310.
Epoch 15/10000, Train RMSE: 0.8026, Val RMSE: 0.9348.
Epoch 16/10000, Train RMSE: 0.7964, Val RMSE: 0.9359.
Epoch 17/10000, Train RMSE: 0.7903, Val RMSE: 0.9377.
Epoch 18/10000, Train RMSE: 0.7843, Val RMSE: 0.9427.
Epoch 19/10000, Train RMSE: 0.7787, V

In [21]:
ranker_no_interactions = RankerWithFeatures(
    users_num=max_user,
    movies_num=max_movie,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
    genres_num=genres_num,
    add_interactions=False,
)

In [ ]:
train_ranker(
    model=ranker_no_interactions,
    train_loader=train_loader,
    val_loader=val_loader,
    var_key_map=var_key_map,
    epochs=10_000,
    learning_rate=0.002,
)

Epoch 1/10000, Train RMSE: 0.9667, Val RMSE: 0.9652.
Epoch 2/10000, Train RMSE: 0.9239, Val RMSE: 0.9545.
Epoch 3/10000, Train RMSE: 0.9098, Val RMSE: 0.9428.
Epoch 4/10000, Train RMSE: 0.8965, Val RMSE: 0.9345.
Epoch 5/10000, Train RMSE: 0.8881, Val RMSE: 0.9306.
Epoch 6/10000, Train RMSE: 0.8830, Val RMSE: 0.9274.
Epoch 7/10000, Train RMSE: 0.8787, Val RMSE: 0.9229.
Epoch 8/10000, Train RMSE: 0.8751, Val RMSE: 0.9220.
Epoch 9/10000, Train RMSE: 0.8721, Val RMSE: 0.9199.
Epoch 10/10000, Train RMSE: 0.8697, Val RMSE: 0.9169.
Epoch 11/10000, Train RMSE: 0.8670, Val RMSE: 0.9165.
Epoch 12/10000, Train RMSE: 0.8648, Val RMSE: 0.9149.
Epoch 13/10000, Train RMSE: 0.8624, Val RMSE: 0.9150.
Epoch 14/10000, Train RMSE: 0.8603, Val RMSE: 0.9138.
Epoch 15/10000, Train RMSE: 0.8579, Val RMSE: 0.9132.
Epoch 16/10000, Train RMSE: 0.8557, Val RMSE: 0.9130.
Epoch 17/10000, Train RMSE: 0.8536, Val RMSE: 0.9116.
Epoch 18/10000, Train RMSE: 0.8514, Val RMSE: 0.9109.
Epoch 19/10000, Train RMSE: 0.8491, V

In [309]:
base_ranker = BaseRanker(
    users_num=max_user,
    movies_num=max_movie,
)

train_ranker(
    model=base_ranker,
    train_loader=train_loader,
    val_loader=val_loader,
    var_key_map={
        "user_ids": "user_id", 
        "movie_ids": "movie_id",
    },
    epochs=10_000,
    learning_rate=0.002,
)

Epoch 1/10000, Train RMSE: 0.9704, Val RMSE: 0.9699.
Epoch 2/10000, Train RMSE: 0.9286, Val RMSE: 0.9608.
Epoch 3/10000, Train RMSE: 0.9184, Val RMSE: 0.9533.
Epoch 4/10000, Train RMSE: 0.9061, Val RMSE: 0.9461.
Epoch 5/10000, Train RMSE: 0.8956, Val RMSE: 0.9360.
Epoch 6/10000, Train RMSE: 0.8889, Val RMSE: 0.9313.
Epoch 7/10000, Train RMSE: 0.8843, Val RMSE: 0.9286.
Epoch 8/10000, Train RMSE: 0.8807, Val RMSE: 0.9251.
Epoch 9/10000, Train RMSE: 0.8779, Val RMSE: 0.9235.
Epoch 10/10000, Train RMSE: 0.8756, Val RMSE: 0.9225.
Epoch 11/10000, Train RMSE: 0.8736, Val RMSE: 0.9207.
Epoch 12/10000, Train RMSE: 0.8716, Val RMSE: 0.9203.
Epoch 13/10000, Train RMSE: 0.8698, Val RMSE: 0.9201.
Epoch 14/10000, Train RMSE: 0.8681, Val RMSE: 0.9210.
Epoch 15/10000, Train RMSE: 0.8663, Val RMSE: 0.9182.
Epoch 16/10000, Train RMSE: 0.8645, Val RMSE: 0.9203.
Epoch 17/10000, Train RMSE: 0.8629, Val RMSE: 0.9195.
Epoch 18/10000, Train RMSE: 0.8614, Val RMSE: 0.9194.
Epoch 19/10000, Train RMSE: 0.8598, V

In [26]:
outer_dataset = Ratings(outer_val_df, genres_cols, is_training=False)

outer_loader = DataLoader(
    outer_dataset, 
    batch_size=1024, 
    shuffle=False,
)


def report_per_user(
        model: nn.Module,
        dataloader: DataLoader[dict[str, torch.Tensor]],
        device: torch.device,
        var_key_map: dict[str, str],
) -> tuple[dict[int, list[float]], dict[int, list[float]]]:
    per_user_preds: dict[int, list[float]] = defaultdict(list)
    per_user_trues: dict[int, list[float]] = defaultdict(list)
    
    with torch.no_grad():
        for batch in dataloader:
            feature_tensors = move_tensors(var_key_map, batch, device)
                
            y_pred = model(**feature_tensors).cpu().numpy()
            y_true = batch["rating"].cpu().numpy()
            user_ids = batch["user_id"].cpu().numpy()
            
            for user_id, pred_value, true_value in zip(user_ids, y_pred, y_true):
                per_user_preds[user_id].append(pred_value)
                per_user_trues[user_id].append(true_value + 5.0)
    
    return per_user_preds, per_user_trues

def count_ndcg_per_user(
    per_user_preds: dict[int, list[float]],
    per_user_trues: dict[int, list[float]],
    recommend_num: int,
):
    ndcg_scores = []
    
    for user_id in per_user_preds:
        if len(per_user_preds[user_id]) > 1:
            user_true = np.array([per_user_trues[user_id]])
            user_pred = np.array([per_user_preds[user_id]])
            
            score = ndcg_score(user_true, user_pred, k=recommend_num)
            ndcg_scores.append(score)
    
    return ndcg_scores


def evaluate_mean_ndcg(
    model: nn.Module,
    dataloader: DataLoader[dict[str, torch.Tensor]],
    var_key_map: dict[str, str],
    device: torch.device,
    recommend_num: int = 5,
) -> float:
    model.eval()
    
    per_user_preds, per_user_trues = report_per_user(
        model, 
        dataloader, 
        device,
        var_key_map,
    )

    ndcg_scores = count_ndcg_per_user(
        per_user_preds,
        per_user_trues,
        recommend_num,
    )
            
    return np.mean(ndcg_scores) if ndcg_scores else 0.0


In [ ]:
evaluate_mean_ndcg(
    ranker_no_interactions,
    outer_loader,
    var_key_map,
    torch.device("cpu"),
    recommend_num=5,
)

np.float64(0.9473999408288181)

In [322]:
from recommender_fm import count_baseline_score

count_baseline_score(
    train_df.drop(columns="rating"),
    outer_val_df.drop(columns="rating"),
    train_df["rating"],
    outer_val_df["rating"],
    recommend_num=5,
)

np.float64(0.849373442865673)

In [ ]:
class PairwiseRatings(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        genre_cols: list[str],
        is_training: bool = True,
    ):
        self.is_training = is_training
        
        self.user_ids = torch.tensor(df["user_id"].values, dtype=torch.long)
        self.movie_ids = torch.tensor(df["movie_id"].values, dtype=torch.long)
        self.ages = torch.tensor(df["age"].values, dtype=torch.long)
        self.genders = torch.tensor(df["gender"].values, dtype=torch.long)
        self.jobs = torch.tensor(df["occupation"].values, dtype=torch.long)
        self.genres = torch.tensor(df[genre_cols].values, dtype=torch.float32)

        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32) 
        
        self.user_to_indices = df.groupby("user_id").indices

    def __len__(self) -> int:
        return len(self.user_ids)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        user_id = self.user_ids[idx].item()
        
        user_indices = self.user_to_indices[user_id]
        idx2 = np.random.choice(user_indices)
        
        r1 = self.ratings[idx]
        r2 = self.ratings[idx2]
        
        if r1 >= r2:
            pos_idx, neg_idx = idx, idx2
        else:
            pos_idx, neg_idx = idx2, idx
            
        rating_diff = self.ratings[pos_idx] - self.ratings[neg_idx]
        
        user_tensor = torch.tensor(user_id, dtype=torch.long)
        # if self.is_training and torch.rand(1).item() < 0.05:
            # user_tensor = torch.tensor(0, dtype=torch.long)
            
        return {
            "user_id": user_tensor,
            "age": self.ages[pos_idx],
            "gender": self.genders[pos_idx],
            "job": self.jobs[pos_idx],
            "pos_movie_id": self.movie_ids[pos_idx],
            "pos_genres": self.genres[pos_idx],
            "neg_movie_id": self.movie_ids[neg_idx],
            "neg_genres": self.genres[neg_idx],
            "rating_diff": rating_diff
        }


def _run_epoch(
    model: nn.Module,
    dataloader: DataLoader[dict[str, torch.Tensor]],
    pos_key_map: dict[str, str],
    neg_key_map: dict[str, str],
    device: torch.device,
    is_train: bool,
    optimizer: optim.Optimizer | None = None,
) -> float:
    if is_train:
        model.train()
    else:
        model.eval()
    
    epoch_loss = 0.0
    dataset_len = 0
    
    with torch.set_grad_enabled(is_train):
        for batch in dataloader:
            pos_tensors = move_tensors(pos_key_map, batch, device)
            neg_tensors = move_tensors(neg_key_map, batch, device)
            rating_diff = batch["rating_diff"].to(device)

            combined_tensors = {}
            for model_var in pos_tensors:
                combined_tensors[model_var] = torch.cat(
                    [pos_tensors[model_var], neg_tensors[model_var]],
                    dim=0,
                )

            y_combined = model(**combined_tensors)
            y_pos, y_neg = torch.chunk(y_combined, 2, dim=0)
            
            base_loss = torch.relu(1.0 - (y_pos - y_neg))
            loss = torch.mean(rating_diff * base_loss)
            
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
            epoch_loss += loss.item() * rating_diff.size(0)
            dataset_len += rating_diff.size(0)
            
    return epoch_loss / dataset_len


def learn_to_rank(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    pos_key_map: dict[str, str],
    neg_key_map: dict[str, str],
    epochs=10_000,
    learning_rate=0.002,
    patience=10,
    tol=1e-4,
    weight_decay=1e-5,
    device_name: str = "cpu",
) -> None:
    
    device = torch.device(device_name)
    model = model.to(device)
    
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    
    best_val_loss = np.inf
    to_beat_loss = np.inf
    no_improve_count = 0
    best_weights = None
    
    for epoch in range(epochs):
        avg_train_loss = _run_epoch(
            model=model,
            dataloader=train_loader,
            pos_key_map=pos_key_map,
            neg_key_map=neg_key_map,
            device=device,
            is_train=True,
            optimizer=optimizer,
        )
        
        avg_val_loss = _run_epoch(
            model=model,
            dataloader=val_loader,
            pos_key_map=pos_key_map,
            neg_key_map=neg_key_map,
            device=device,
            is_train=False,
        )
        
        print(
            f"Epoch {epoch+1}/{epochs}, "
            f"Train Loss: {avg_train_loss:.4f}, "
            f"Val Loss: {avg_val_loss:.4f}."
        )
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_weights = copy.deepcopy(model.state_dict())

        if avg_val_loss < (to_beat_loss - tol):
            to_beat_loss = avg_val_loss
            no_improve_count = 0
        else:
            no_improve_count += 1
            
        if no_improve_count >= patience:
            break
    
    if best_weights is not None:
        model.load_state_dict(best_weights)
        print(
            "Best weights restored. "
            f"Validation loss is {best_val_loss:.4f}"
        )

In [18]:
pos_key_map = {
    "user_ids": "user_id", 
    "movie_ids": "pos_movie_id", 
    "ages": "age", 
    "genders": "gender", 
    "jobs": "job", 
    "genres": "pos_genres",
}

neg_key_map = {
    "user_ids": "user_id", 
    "movie_ids": "neg_movie_id",
    "ages": "age", 
    "genders": "gender", 
    "jobs": "job", 
    "genres": "neg_genres",
}

In [19]:
train_dataset = PairwiseRatings(train_df, genres_cols, is_training=True)
val_dataset = PairwiseRatings(val_df, genres_cols, is_training=False)

train_loader = DataLoader(
    train_dataset, 
    batch_size=1024, 
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=1024, 
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

In [24]:
ranker_with_features = RankerWithFeatures(
    users_num=max_user,
    movies_num=max_movie,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
    genres_num=genres_num,
    add_interactions=False,
)

learn_to_rank(
    model=ranker_no_interactions,
    train_loader=train_loader,
    val_loader=val_loader,
    pos_key_map=pos_key_map,
    neg_key_map=neg_key_map,
    epochs=10_000,
    learning_rate=0.0005,
    weight_decay=1e-3,
)

Epoch 1/10000, Train Loss: 0.2523, Val Loss: 0.8481.
Epoch 2/10000, Train Loss: 0.2429, Val Loss: 0.8136.
Epoch 3/10000, Train Loss: 0.2329, Val Loss: 0.8113.
Epoch 4/10000, Train Loss: 0.2417, Val Loss: 0.8006.
Epoch 5/10000, Train Loss: 0.2115, Val Loss: 0.7764.
Epoch 6/10000, Train Loss: 0.2298, Val Loss: 0.7918.
Epoch 7/10000, Train Loss: 0.2150, Val Loss: 0.8407.
Epoch 8/10000, Train Loss: 0.1863, Val Loss: 0.8143.
Epoch 9/10000, Train Loss: 0.1739, Val Loss: 0.8149.
Epoch 10/10000, Train Loss: 0.1938, Val Loss: 0.8224.
Epoch 11/10000, Train Loss: 0.1847, Val Loss: 0.8123.
Epoch 12/10000, Train Loss: 0.1906, Val Loss: 0.8296.
Epoch 13/10000, Train Loss: 0.1793, Val Loss: 0.7756.
Epoch 14/10000, Train Loss: 0.1842, Val Loss: 0.7991.
Epoch 15/10000, Train Loss: 0.1668, Val Loss: 0.7924.
Epoch 16/10000, Train Loss: 0.1718, Val Loss: 0.7781.
Epoch 17/10000, Train Loss: 0.1750, Val Loss: 0.8073.
Epoch 18/10000, Train Loss: 0.1676, Val Loss: 0.8251.
Epoch 19/10000, Train Loss: 0.1637, V

In [29]:
evaluate_mean_ndcg(
    ranker_no_interactions,
    outer_loader,
    var_key_map,
    torch.device("cpu"),
    recommend_num=5,
)

np.float64(0.9343539534352637)